# 02 · Preprocesamiento y Features
## Segmentación Inteligente de Pacientes — Comfama

| | |
|---|---|
| **Propósito** | Limpiar, imputar y tipar el bloque L:AB para dejar una tabla lista para el modelo. |
| **Entrada** | Tabla Delta `<catalog>.<schema>.fenotipo_cardio_ingreso_raw` (notebook 01). |
| **Salida** | Tabla Delta `<catalog>.<schema>.fenotipo_features`. |
| **Siguiente paso** | `03_entrenamiento_clustering`. |

---
### 🧭 Reglas de transformación (trazabilidad)
| Tipo de variable | Regla aplicada | Justificación |
|---|---|---|
| Binarias `hab_*`, `med_*` | a numérico → `>0 ⇒ 1`, NaN → `0` | ausencia = "no aplica / no reportado" |
| `habitos_alim_cantidad` | a entero ≥ 0, NaN → `0` | conteo de hábitos |
| Categóricas `actividad_*`, `estres_*` | `SIN DATO`/vacío/NaN → `"Desconocido"`; se quita `;` final | preservar la ausencia como categoría informativa |
| `habitos_alim_texto` (L), `medicamentos_texto` (Y) | **se descartan** | redundantes con las binarias N–S y Z–AB |

> **Decisión de diseño:** el **escalado** y el ***one-hot*** se dejan **dentro del Pipeline del modelo** (notebook 03) para evitar fuga de información y versionar la transformación junto con el modelo en MLflow. Aquí solo limpiamos e imputamos.

### Paso 1 · Parámetros y carga de la capa *raw*

In [ ]:
dbutils.widgets.text("catalog", "main", "Catálogo")
dbutils.widgets.text("schema", "fenotipo", "Esquema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

RAW_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_cardio_ingreso_raw"
FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_features"

import pandas as pd
import numpy as np

pdf = spark.table(RAW_TABLE).toPandas()
print(f"Filas cargadas: {len(pdf):,}")

### Paso 2 · Catálogo de variables por rol
Declaramos explícitamente qué variables son numéricas, categóricas y cuáles se descartan. Este catálogo se **reutiliza idéntico** en los notebooks 03 y 04 (contrato de features).

In [ ]:
NUMERIC_FEATURES = [
    "habitos_alim_cantidad",
    "hab_dejar_endulzar", "hab_dejar_reposteria", "hab_incorporar_vegetales",
    "hab_ajustar_carbohidratos", "hab_aumentar_proteina", "hab_otros",
    "med_hipoglicemiantes", "med_hipolipemiantes", "med_ninguno",
]
CATEGORICAL_FEATURES = [
    "actividad_tipo", "actividad_duracion", "actividad_frecuencia",
    "estres_altas_cargas", "estres_tecnica_manejo",
]
DROP_COLS = ["habitos_alim_texto", "medicamentos_texto"]
ID_COLS = ["id", "fecha_atencion", "grupos"]

print(f"Numéricas: {len(NUMERIC_FEATURES)} | Categóricas: {len(CATEGORICAL_FEATURES)}")

### Paso 3 · Limpieza e imputación
Aplicamos las reglas de la tabla superior. Al finalizar **no deben quedar nulos** en las variables del modelo.

In [ ]:
df = pdf.drop(columns=[c for c in DROP_COLS if c in pdf.columns]).copy()

# --- Numéricas / binarias: a número, NaN -> 0 ---
for c in NUMERIC_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    if c != "habitos_alim_cantidad":
        df[c] = (df[c] > 0).astype(int)          # binaria {0,1}
    else:
        df[c] = df[c].clip(lower=0).astype(int)  # conteo >=0

# --- Categóricas: "SIN DATO"/vacío/NaN -> "Desconocido" ---
def limpiar_categoria(serie: pd.Series) -> pd.Series:
    s = serie.astype("string").str.strip()
    s = s.replace({"SIN DATO": pd.NA, "": pd.NA, "nan": pd.NA, "None": pd.NA})
    s = s.str.rstrip(";").str.strip()   # quita ';' final del export de formularios
    return s.fillna("Desconocido")

for c in CATEGORICAL_FEATURES:
    df[c] = limpiar_categoria(df[c])

### Paso 4 · Validación post-limpieza (control de calidad)

In [ ]:
nulos = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().sum()
assert nulos.sum() == 0, f"Quedan nulos: \n{nulos[nulos>0]}"
print("✅ Sin nulos en las variables del modelo.\n")

print("Categorías finales por variable:")
for c in CATEGORICAL_FEATURES:
    print(f"  {c}: {sorted(df[c].unique().tolist())}")

display(df.head(20))

### Paso 5 · Persistencia (capa *features* / plata)
Guardamos la tabla lista para modelar. Es la **única entrada** del notebook de entrenamiento.

In [ ]:
for c in df.columns:
    if df[c].dtype == "object" or str(df[c].dtype) == "string":
        df[c] = df[c].astype(str)

sdf = spark.createDataFrame(df)
(sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURES_TABLE))
print(f"✅ Features guardadas: {FEATURES_TABLE}  ({sdf.count():,} filas, {len(df.columns)} columnas)")

---
**Resumen del notebook 02:** datos limpios, imputados y tipados en `fenotipo_features`.
➡️ Continúa en **`03_entrenamiento_clustering`**.